In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!kaggle kernels output manvidhamija/ids-18 -p /path/to/dest

Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_1.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_10.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_11.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_12.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_13.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_14.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_15.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_16.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_17.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_18.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_19.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cleaned/clean_chunk_2.csv
Output file downloaded to /path/to/dest/CIC_IDS2018_Cl

In [3]:
!kaggle kernels output manvidhamija/ids-18-2 -p /path/to/dest

Output file downloaded to /path/to/dest/Transformer_Pipeline/feature_columns.pkl
Output file downloaded to /path/to/dest/Transformer_Pipeline/label_encoder.pkl
Output file downloaded to /path/to/dest/Transformer_Pipeline/standard_scaler.pkl
Output file downloaded to /path/to/dest/best_transformer_model.pth
Output file downloaded to /path/to/dest/feature_columns.pkl
Output file downloaded to /path/to/dest/label_encoder.pkl
Output file downloaded to /path/to/dest/standard_scaler.pkl
Kernel log downloaded to /path/to/dest/ids-18-2.log 


In [4]:
# ============================================================
# Cell 1 - Imports
# ============================================================

import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

In [5]:
# ============================================================
# Cell 2 - Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Dataset Paths
# ------------------------------------------------------------

INPUT_DIR = "/path/to/dest/CIC_IDS2018_HTRE_Final"

# ------------------------------------------------------------
# Dataset Constants
# ------------------------------------------------------------

TIMESTAMP_COLUMN = "Timestamp"
LABEL_COLUMN = "Label"

print("=" * 70)
print("Configuration Loaded")
print("=" * 70)
print(f"Dataset : {INPUT_DIR}")
print(f"Seed    : {SEED}")

Configuration Loaded
Dataset : /path/to/dest/CIC_IDS2018_HTRE_Final
Seed    : 42


In [6]:
# ============================================================
# Cell 3 - Dataset Discovery
# ============================================================

print("=" * 70)
print("Dataset Discovery")
print("=" * 70)

# ------------------------------------------------------------
# Discover HTRE Dataset Files
# ------------------------------------------------------------

dataset_files = sorted(
    [
        file
        for file in os.listdir(INPUT_DIR)
        if file.endswith(".csv")
    ]
)

print(f"Total CSV Files : {len(dataset_files)}")
print()

print("First 10 Files")
for file in dataset_files[:10]:
    print(file)

if len(dataset_files) > 10:
    print("...")

print()
print("Last File :", dataset_files[-1])

Dataset Discovery
Total CSV Files : 32

First 10 Files
clean_chunk_1.csv
clean_chunk_10.csv
clean_chunk_11.csv
clean_chunk_12.csv
clean_chunk_13.csv
clean_chunk_14.csv
clean_chunk_15.csv
clean_chunk_16.csv
clean_chunk_17.csv
clean_chunk_18.csv
...

Last File : clean_chunk_9.csv


In [7]:
# ============================================================
# Cell 4 - Dataset Summary
# ============================================================

print("=" * 70)
print("Computing Dataset Statistics...")
print("=" * 70)

total_rows = 0

dataset_summary = []

for file in tqdm(dataset_files):

    file_path = os.path.join(INPUT_DIR, file)

    n_rows = sum(1 for _ in open(file_path)) - 1   # subtract header

    dataset_summary.append(
        {
            "File": file,
            "Rows": n_rows
        }
    )

    total_rows += n_rows

dataset_summary = pd.DataFrame(dataset_summary)

print()
print("=" * 70)
print("Dataset Summary")
print("=" * 70)

print(f"Total Files : {len(dataset_summary)}")
print(f"Total Rows  : {total_rows:,}")

print()
print(dataset_summary.head())

Computing Dataset Statistics...


  0%|          | 0/32 [00:00<?, ?it/s]


Dataset Summary
Total Files : 32
Total Rows  : 15,730,971

                 File    Rows
0   clean_chunk_1.csv  498028
1  clean_chunk_10.csv  500000
2  clean_chunk_11.csv  500000
3  clean_chunk_12.csv  500000
4  clean_chunk_13.csv  500000


In [8]:
# ============================================================
# Cell 5 - Data Inventory
# ============================================================

print("=" * 70)
print("Building Data Inventory...")
print("=" * 70)

inventory = []

for file in tqdm(dataset_files):

    file_path = os.path.join(INPUT_DIR, file)

    df = pd.read_csv(
        file_path,
        usecols=[TIMESTAMP_COLUMN, LABEL_COLUMN]
    )

    # Parse timestamps
    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    df = df[df[TIMESTAMP_COLUMN].notna()]

    inventory.append({

        "File": file,

        "Rows": len(df),

        "Start Time": df[TIMESTAMP_COLUMN].min(),

        "End Time": df[TIMESTAMP_COLUMN].max(),

        "Days": df[TIMESTAMP_COLUMN].dt.date.nunique(),

        "Classes": df[LABEL_COLUMN].nunique()

    })

    del df
    gc.collect()

inventory = pd.DataFrame(inventory)

print()
print("=" * 70)
print("Data Inventory")
print("=" * 70)

display(inventory)

print()
print(f"Total Files : {len(inventory)}")
print(f"Total Rows  : {inventory['Rows'].sum():,}")
print(f"Total Days  : {inventory['Days'].sum()}")

Building Data Inventory...


  0%|          | 0/32 [00:00<?, ?it/s]


Data Inventory


,File,Rows,Start Time,End Time,Days,Classes
0,clean_chunk_1.csv,498028,2018-03-02 01:00:00,2018-03-02 12:58:43,1,1
1,clean_chunk_10.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
2,clean_chunk_11.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
3,clean_chunk_12.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
4,clean_chunk_13.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
5,clean_chunk_14.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
6,clean_chunk_15.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
7,clean_chunk_16.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
8,clean_chunk_17.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1
9,clean_chunk_18.csv,500000,2018-02-20 01:00:00,2018-02-20 12:59:59,1,1



Total Files : 32
Total Rows  : 15,730,971
Total Days  : 47


In [9]:
# ============================================================
# Cell 6 - Day Inventory
# ============================================================

print("=" * 70)
print("Building Day Inventory...")
print("=" * 70)

day_inventory = []

for file in tqdm(dataset_files):

    file_path = os.path.join(INPUT_DIR, file)

    df = pd.read_csv(
        file_path,
        usecols=[TIMESTAMP_COLUMN, LABEL_COLUMN]
    )

    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    df = df[df[TIMESTAMP_COLUMN].notna()]

    for day, group in df.groupby(df[TIMESTAMP_COLUMN].dt.date):

        day_inventory.append({

            "File": file,

            "Date": str(day),

            "Rows": len(group),

            "Classes": sorted(group[LABEL_COLUMN].unique())

        })

    del df
    gc.collect()

day_inventory = pd.DataFrame(day_inventory)

print()
print("=" * 70)
print("Day Inventory")
print("=" * 70)

display(day_inventory)

print()
print("Unique Days :", day_inventory["Date"].nunique())

Building Day Inventory...


  0%|          | 0/32 [00:00<?, ?it/s]


Day Inventory


,File,Date,Rows,Classes
0,clean_chunk_1.csv,2018-03-02,498028,[Benign]
1,clean_chunk_10.csv,2018-02-20,500000,[Benign]
2,clean_chunk_11.csv,2018-02-20,500000,[Benign]
3,clean_chunk_12.csv,2018-02-20,500000,[Benign]
4,clean_chunk_13.csv,2018-02-20,500000,[Benign]
5,clean_chunk_14.csv,2018-02-20,500000,[Benign]
6,clean_chunk_15.csv,2018-02-20,500000,[Benign]
7,clean_chunk_16.csv,2018-02-20,500000,[Benign]
8,clean_chunk_17.csv,2018-02-20,500000,[Benign]
9,clean_chunk_18.csv,2018-02-20,500000,[Benign]



Unique Days : 15


In [27]:
# ============================================================
# Cell 7 - Load Pipeline Artifacts
# ============================================================

import joblib

print("=" * 70)
print("Loading Pipeline Artifacts...")
print("=" * 70)

# ------------------------------------------------------------
# Artifact Paths
# ------------------------------------------------------------

FEATURE_COLUMNS_PATH = "/path/to/dest/Transformer_Pipeline/feature_columns.pkl"
LABEL_ENCODER_PATH = "/path/to/dest/Transformer_Pipeline/label_encoder.pkl"
STANDARD_SCALER_PATH = "/path/to/dest/Transformer_Pipeline/standard_scaler.pkl"

# ------------------------------------------------------------
# Load Feature Columns
# ------------------------------------------------------------

feature_columns = joblib.load(FEATURE_COLUMNS_PATH)

# ------------------------------------------------------------
# Load Label Encoder
# ------------------------------------------------------------

label_encoder = joblib.load(LABEL_ENCODER_PATH)
scaler = joblib.load(STANDARD_SCALER_PATH)

NUM_CLASSES = len(label_encoder.classes_)

print()

print("Artifacts Loaded Successfully")

print(f"Feature Columns : {len(feature_columns)}")

print(f"Classes         : {NUM_CLASSES}")

print()

print("Feature Preview")

print(feature_columns[:10])

print()

print("Classes")

for i, cls in enumerate(label_encoder.classes_):

    print(f"{i:2d} : {cls}")

Loading Pipeline Artifacts...

Artifacts Loaded Successfully
Feature Columns : 43
Classes         : 14

Feature Preview
['Init Fwd Win Byts', 'Fwd Seg Size Min', 'Fwd Header Len', 'Dst Port', 'Flow Duration', 'Flow IAT Max', 'Fwd Pkts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Init Bwd Win Byts']

Classes
 0 : Benign
 1 : Brute Force -Web
 2 : Brute Force -XSS
 3 : DDOS attack-HOIC
 4 : DDOS attack-LOIC-UDP
 5 : DDoS attacks-LOIC-HTTP
 6 : DoS attacks-GoldenEye
 7 : DoS attacks-Hulk
 8 : DoS attacks-SlowHTTPTest
 9 : DoS attacks-Slowloris
10 : FTP-BruteForce
11 : Infilteration
12 : SQL Injection
13 : SSH-Bruteforce


In [11]:
DATA_CONFIG = {
    "window_size": 20
}

In [12]:
# ============================================================
# Cell 8 - Generate Sequence Dataset
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view

print("=" * 70)
print("Generating Sequence Dataset...")
print("=" * 70)

# ------------------------------------------------------------
# Output Directory
# ------------------------------------------------------------

SEQUENCE_DIR = "/path/to/dest/processed_sequences"

os.makedirs(SEQUENCE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Sliding Window Function
# ------------------------------------------------------------

def create_sequences(X, y, window_size):

    if len(X) < window_size:
        return None, None

    X_seq = sliding_window_view(
        X,
        window_shape=window_size,
        axis=0
    )

    X_seq = np.transpose(
        X_seq,
        (0, 2, 1)
    )

    y_seq = y[window_size - 1:]

    return (
        X_seq.astype(np.float32),
        y_seq.astype(np.int64)
    )

# ------------------------------------------------------------
# Generate Sequence Files
# ------------------------------------------------------------

generated_files = []

for _, row in tqdm(day_inventory.iterrows(),
                   total=len(day_inventory)):

    file_path = os.path.join(INPUT_DIR, row["File"])

    df = pd.read_csv(file_path)

    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    # Remove invalid timestamps
    df = df[
        (df[TIMESTAMP_COLUMN].notna()) &
        (df[TIMESTAMP_COLUMN].dt.year == 2018)
    ]

    # Keep only current day
    df = df[
        df[TIMESTAMP_COLUMN].dt.date.astype(str)
        == row["Date"]
    ]

    df = df.sort_values(
        TIMESTAMP_COLUMN
    ).reset_index(drop=True)

    if len(df) < DATA_CONFIG["window_size"]:

        continue

    # --------------------------------------------------------
    # Features & Labels
    # --------------------------------------------------------

    X = df[
        feature_columns
    ].to_numpy(dtype=np.float32)

    y = label_encoder.transform(
        df[LABEL_COLUMN]
    )

    # --------------------------------------------------------
    # Sliding Windows
    # --------------------------------------------------------

    X_seq, y_seq = create_sequences(
        X,
        y,
        DATA_CONFIG["window_size"]
    )

    if X_seq is None:

        continue

    # --------------------------------------------------------
    # Save Torch Dataset
    # --------------------------------------------------------

    save_name = (
        f"seq_"
        f"{row['File'].replace('.csv','')}"
        f"_{row['Date']}.pt"
    )

    save_path = os.path.join(
        SEQUENCE_DIR,
        save_name
    )

    torch.save(

        {

            "X": torch.from_numpy(X_seq),

            "y": torch.from_numpy(y_seq),

            "metadata": {

                "chunk": row["File"],

                "date": row["Date"],

                "num_sequences": len(y_seq),

                "window_size": DATA_CONFIG["window_size"],

                "num_features": len(feature_columns),

                "classes": sorted(
                    df[LABEL_COLUMN].unique().tolist()
                )

            }

        },

        save_path

    )

    generated_files.append(save_name)

    del df
    del X
    del y
    del X_seq
    del y_seq

    gc.collect()

print()

print("=" * 70)
print("Sequence Dataset Generation Complete")
print("=" * 70)

print(f"Generated Files : {len(generated_files)}")

print(f"Output Folder   : {SEQUENCE_DIR}")

Generating Sequence Dataset...


  0%|          | 0/47 [00:00<?, ?it/s]


Sequence Dataset Generation Complete
Generated Files : 41
Output Folder   : /path/to/dest/processed_sequences


In [13]:
import os

print("Exists:", os.path.exists(SEQUENCE_DIR))

print("Files:")
print(os.listdir(SEQUENCE_DIR)[:10])

print("Total files:", len(os.listdir(SEQUENCE_DIR)))

Exists: True
Files:
['seq_clean_chunk_2_2018-03-02.pt', 'seq_clean_chunk_1_2018-03-02.pt', 'seq_clean_chunk_16_2018-02-20.pt', 'seq_clean_chunk_22_2018-02-20.pt', 'seq_clean_chunk_4_2018-02-16.pt', 'seq_clean_chunk_27_2018-02-22.pt', 'seq_clean_chunk_6_2018-02-23.pt', 'seq_clean_chunk_14_2018-02-20.pt', 'seq_clean_chunk_15_2018-02-20.pt', 'seq_clean_chunk_18_2018-02-20.pt']
Total files: 41


In [14]:
import os

total_size = 0

for f in os.listdir(SEQUENCE_DIR):
    total_size += os.path.getsize(os.path.join(SEQUENCE_DIR, f))

print(f"Total Size : {total_size / (1024**3):.2f} GB")

Total Size : 50.51 GB


In [29]:
import glob
import os

for file in glob.glob(os.path.join(FLOW_DIR, "*.pt")):
    os.remove(file)

print("Old tensors deleted.")

Old tensors deleted.


In [30]:
import os
import glob

remaining = glob.glob(os.path.join(FLOW_DIR, "*.pt"))

print(f"Remaining tensor files: {len(remaining)}")

if len(remaining) == 0:
    print("✅ All old tensors successfully deleted.")
else:
    print("❌ Some tensor files still exist:")
    for f in remaining:
        print(os.path.basename(f))

Remaining tensor files: 0
✅ All old tensors successfully deleted.


In [31]:
# ============================================================
# Cell 8 - Generate Flow Tensors
# ============================================================

print("=" * 70)
print("Generating Flow Tensors...")
print("=" * 70)

# ------------------------------------------------------------
# Output Directory
# ------------------------------------------------------------

FLOW_DIR = "/kaggle/working/flow_tensors"

os.makedirs(FLOW_DIR, exist_ok=True)

# ------------------------------------------------------------
# Catalog Records
# ------------------------------------------------------------

catalog_records = []

# ------------------------------------------------------------
# Generate Flow Tensors
# ------------------------------------------------------------

for idx, row in tqdm(day_inventory.iterrows(),
                     total=len(day_inventory)):

    file_path = os.path.join(INPUT_DIR, row["File"])

    df = pd.read_csv(file_path)

    # --------------------------------------------------------
    # Timestamp Processing
    # --------------------------------------------------------

    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    df = df[
        (df[TIMESTAMP_COLUMN].notna()) &
        (df[TIMESTAMP_COLUMN].dt.year == 2018)
    ]

    df = df[
        df[TIMESTAMP_COLUMN].dt.date.astype(str)
        == row["Date"]
    ]

    df = df.sort_values(
        TIMESTAMP_COLUMN
    ).reset_index(drop=True)

    if len(df) == 0:

        continue

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    X = df[
        feature_columns
    ].to_numpy(dtype=np.float32)

    X = scaler.transform(X).astype(np.float32)

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    y = label_encoder.transform(
        df[LABEL_COLUMN]
    )

    # --------------------------------------------------------
    # Convert to Torch
    # --------------------------------------------------------

    X = torch.from_numpy(X)

    y = torch.from_numpy(
        y.astype(np.int64)
    )

    # --------------------------------------------------------
    # Save Tensor
    # --------------------------------------------------------

    tensor_name = f"flow_{idx:04d}.pt"

    save_path = os.path.join(
        FLOW_DIR,
        tensor_name
    )

    torch.save(

        {

            "X": X,

            "y": y,

            "metadata": {

                "file": row["File"],

                "date": row["Date"],

                "num_flows": len(y),

                "num_features": X.shape[1],

                "classes": sorted(
                    df[LABEL_COLUMN].unique().tolist()
                )

            }

        },

        save_path

    )

    # --------------------------------------------------------
    # Catalog Entry
    # --------------------------------------------------------

    catalog_records.append({

        "Tensor File": tensor_name,

        "Source File": row["File"],

        "Date": row["Date"],

        "Flows": len(y),

        "Features": X.shape[1],

        "Classes": ", ".join(
            sorted(df[LABEL_COLUMN].unique())
        )

    })

    del df
    del X
    del y

    gc.collect()

print()

print("=" * 70)
print("Flow Tensor Generation Complete")
print("=" * 70)

print(f"Generated Files : {len(catalog_records)}")

print(f"Output Folder   : {FLOW_DIR}")

Generating Flow Tensors...


  0%|          | 0/47 [00:00<?, ?it/s]


Flow Tensor Generation Complete
Generated Files : 41
Output Folder   : /kaggle/working/flow_tensors


In [32]:
# ============================================================
# Cell 9 - Build Flow Catalog
# ============================================================

flow_catalog = pd.DataFrame(catalog_records)

catalog_path = os.path.join(
    FLOW_DIR,
    "flow_catalog.csv"
)

flow_catalog.to_csv(
    catalog_path,
    index=False
)

print("=" * 70)
print("Flow Catalog Created")
print("=" * 70)

print(flow_catalog.head())

print()

print(f"Catalog Path : {catalog_path}")

print(f"Total Flow Files : {len(flow_catalog)}")

Flow Catalog Created
    Tensor File         Source File        Date   Flows  Features Classes
0  flow_0000.pt   clean_chunk_1.csv  2018-03-02  498028        43  Benign
1  flow_0001.pt  clean_chunk_10.csv  2018-02-20  500000        43  Benign
2  flow_0002.pt  clean_chunk_11.csv  2018-02-20  500000        43  Benign
3  flow_0003.pt  clean_chunk_12.csv  2018-02-20  500000        43  Benign
4  flow_0004.pt  clean_chunk_13.csv  2018-02-20  500000        43  Benign

Catalog Path : /kaggle/working/flow_tensors/flow_catalog.csv
Total Flow Files : 41


In [33]:
# ============================================================
# Cell 10 - Validate Flow Tensors
# ============================================================

print("=" * 70)
print("Validating Flow Tensor Dataset...")
print("=" * 70)

sample = flow_catalog.iloc[0]

sample_path = os.path.join(
    FLOW_DIR,
    sample["Tensor File"]
)

data = torch.load(sample_path)

print()

print("Sample File")

print(sample["Tensor File"])

print()

print("X Shape :", data["X"].shape)

print("y Shape :", data["y"].shape)

print()

print("Metadata")

for k, v in data["metadata"].items():

    print(f"{k:<15} : {v}")

print()

total_flows = flow_catalog["Flows"].sum()

print("=" * 70)

print(f"Total Tensor Files : {len(flow_catalog)}")

print(f"Total Flows        : {total_flows:,}")

print("=" * 70)

Validating Flow Tensor Dataset...

Sample File
flow_0000.pt

X Shape : torch.Size([498028, 43])
y Shape : torch.Size([498028])

Metadata
file            : clean_chunk_1.csv
date            : 2018-03-02
num_flows       : 498028
num_features    : 43
classes         : ['Benign']

Total Tensor Files : 41
Total Flows        : 15,730,957


In [34]:
print("="*70)
print("Notebook 2 Complete")
print("="*70)

print(f"Flow Tensors : {len(flow_catalog)}")
print(f"Total Flows  : {flow_catalog['Flows'].sum():,}")
print(f"Features     : {flow_catalog['Features'].iloc[0]}")
print(f"Classes      : {len(label_encoder.classes_)}")

print()
print("Artifacts Generated")
print("-------------------")
print("flow_tensors/")
print("flow_catalog.csv")

print("="*70)

Notebook 2 Complete
Flow Tensors : 41
Total Flows  : 15,730,957
Features     : 43
Classes      : 14

Artifacts Generated
-------------------
flow_tensors/
flow_catalog.csv


In [35]:
import torch
import os

sample = torch.load(
    os.path.join(FLOW_DIR, "flow_0000.pt"),
    map_location="cpu"
)

X = sample["X"]

print("="*60)
print("Tensor Statistics")
print("="*60)

print("dtype :", X.dtype)
print("shape :", X.shape)

print("Min   :", X.min())
print("Max   :", X.max())
print("Mean  :", X.mean())
print("Std   :", X.std())

Tensor Statistics
dtype : torch.float32
shape : torch.Size([498028, 43])
Min   : tensor(-3.8208)
Max   : tensor(377.5883)
Mean  : tensor(0.0243)
Std   : tensor(0.9633)
